In [1]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import customfunctions as ctf
from scipy import signal
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from brainflow.data_filter import DataFilter, WindowOperations
import mne
import mne_connectivity as mnec
from mne_connectivity.viz import plot_sensors_connectivity

from itertools import combinations, product


In [2]:
# Working path
folder = 'Rawdata'
cwd = os.getcwd()
folderpath = os.path.join(cwd,folder)
print(folderpath)


c:\Users\josee\OneDrive\Documentos\GitHub\NeuroHAD2025\FILAnalysis\Rawdata


In [3]:
# Sessions of interest
sessions = [2, 12, 14, 21, 27, 28, 31, 39, 41, 42]
# Stages
stages = ["baseline", "shared", "individual"]
# Se guardan los datos de las sesiones en un diccionario
preprodata = {
    "p1": {"baseline": [], "shared": [], "individual": []},
    "p2": {"baseline": [], "shared": [], "individual": []},
}

for stage in stages:
    p1holder = []
    p2holder = []
    for sn in sessions:
        p1 = []
        p2 = []
        pattern = f"S{sn}R*"
        snpath = glob.glob(os.path.join(folderpath, pattern))[0]

        p1 = (
            pd.read_csv(os.path.join(snpath, "processed", f"p1_{stage}.csv"))
            .drop(["Unnamed: 0","board_ts","unix_ts"], axis=1)
            .replace(["Unnamed: 0","board_ts","unix_ts"], np.nan)
            .dropna()
            .apply(pd.to_numeric, axis=0)
        )
        if stage == 'baseline':
            p1 = p1.iloc[2561:25601,:]
        else:
            p1 = p1.iloc[2561:64001,:]
        p1.subtract(p1.mean(axis=1), axis=0) # Common mean reference
        p1.insert(loc=0, column="Session", value=np.ones([p1.shape[0], 1]) * sn)
        p1holder.append(p1)
        p2 = (
            pd.read_csv(os.path.join(snpath, "processed", f"p2_{stage}.csv"))
            .drop(["Unnamed: 0","board_ts","unix_ts"], axis=1)
            .replace(["Unnamed: 0","board_ts","unix_ts"], np.nan)
            .dropna()
            .apply(pd.to_numeric, axis=0)
        )
        if stage == 'baseline':
            p2 = p2.iloc[2561:25601,:]
        else:
            p2 = p2.iloc[2561:64001,:]
        p2.subtract(p2.mean(axis=1), axis=0)
        p2.insert(loc=0, column="Session", value=np.ones([p2.shape[0], 1]) * sn)
        p2holder.append(p2)
    p1holder_df = pd.concat(p1holder, ignore_index=True)
    p2holder_df = pd.concat(p2holder, ignore_index=True)
    preprodata["p1"].update({stage:p1holder_df})
    preprodata["p2"].update({stage:p2holder_df})

In [4]:
preprodata['p2']['baseline'][ preprodata['p1']['baseline']['Session'] == 41 ]

,Session,TP9,AF7,AF8,TP10
184320,41.0,6.283599,9.033713,-5.895783,-25.784886
184321,41.0,3.790505,7.823831,-9.203100,-33.227550
184322,41.0,-5.006978,4.468822,-9.762994,-32.378601
184323,41.0,-17.859640,-0.774919,-7.069969,-26.946536
184324,41.0,-28.629580,-6.175805,-4.470735,-23.317676
...,...,...,...,...,...
207355,41.0,38.136049,-30.470081,-17.548078,39.400611
207356,41.0,60.011499,-22.421810,-39.473445,58.071335
207357,41.0,45.823451,8.910796,-33.432287,46.696319
207358,41.0,1.850072,25.084634,-1.048385,5.365356


In [5]:
fs = 256
winsz = 1 
stages = ["baseline", "shared", "individual"]
channels = ["TP9", "TP10", "AF7", "AF8"]
BANDS = [
    ("theta", 4.0, 8.0),
    ("alpha", 8.0, 13.0),
    ("beta", 13.0, 30.0),
    ("gamma", 30.0, 50.0),
]
feats = ['Theta','Alpha','Beta','Gamma', 'TEI']
featcols = [f"{ch}_{ft}" for ch in channels for ft in feats] + ['AlphaAsymmetry']
# Se guardan los datos de las sesiones en un diccionario
metricdata = {
    "p1": {"baseline": [], "shared": [], "individual": []},
    "p2": {"baseline": [], "shared": [], "individual": []},
}

In [6]:
def bandpowers_per_window(winddata, fs, bands=BANDS):
    n_win, win_len = winddata.shape
    n_bands = len(bands)
    out = np.empty((n_win, n_bands), dtype=float)

    nfft = DataFilter.get_nearest_power_of_two(win_len)

    for w in range(n_win):
        sig = winddata[w, :].astype(np.float64)
        psd = DataFilter.get_psd_welch(
            sig,
            nfft,
            nfft // 2,           # overlap 50%
            fs,
            WindowOperations.HANNING.value
        )
        for bi, (_, fmin, fmax) in enumerate(bands):
            out[w, bi] = DataFilter.get_band_power(psd, fmin, fmax)

    return out

def extract_session_bandpowers(sessiondata, channels, fs, winsz, eps=1e-12):
    # Calcular bandpowers para cada canal
    bandmats = []
    for ch in channels:
        sig = sessiondata[ch].to_numpy()
        wind = ctf.timewindow(sig, fs, winsz, 0)     
        bp = bandpowers_per_window(wind, fs)          
        bandmats.append(bp)


    # TEI por canal 
    tei_list = []
    for bp in bandmats:
        theta = bp[:, 0]
        alpha = bp[:, 1]
        beta  = bp[:, 2]
        tei = beta / (theta + alpha + eps)            
        tei_list.append(tei[:, None])                 

    # Alpha Asym
    alpha_AF7 = bandmats[2][:, 1]  
    alpha_AF8 = bandmats[3][:, 1]   
    alpha_asym = (np.log(alpha_AF8 + eps) - np.log(alpha_AF7 + eps))[:, None]


    rows = []

    for i, ch in enumerate(channels):
        bp_ch = bandmats[i]                   
        tei_ch = tei_list[i]                
        block = np.concatenate([bp_ch, tei_ch], axis=1) 
        rows.append(block)

    # Unir todos los canales en columnas: (n_win, 5*4 = 20)
    per_channel = np.concatenate(rows, axis=1)

    # Agregar alpha asym al final
    out = np.concatenate([per_channel, alpha_asym], axis=1)

    return out

for stage in stages:
    stagedata1 = preprodata['p1'][stage].copy()
    stagedata2 = preprodata['p2'][stage].copy()
    stageholder1 = []
    stageholder2 = []
    for sn in sessions:
        sessiondata1 = stagedata1[stagedata1["Session"]==sn].drop("Session",axis=1)
        bandmat1 = extract_session_bandpowers(sessiondata1, channels, fs, winsz)
        bandmat1_df = pd.DataFrame(bandmat1,columns=featcols,dtype=float)
        bandmat1_df.insert(loc=0, column="Session", value=np.ones([bandmat1_df.shape[0], 1]) * sn)
        stageholder1.append(bandmat1_df)


        sessiondata2 = stagedata2[stagedata2["Session"]==sn].drop("Session",axis=1)
        bandmat2 = extract_session_bandpowers(sessiondata2, channels, fs, winsz)
        bandmat2_df = pd.DataFrame(bandmat2,columns=featcols,dtype=float)
        bandmat2_df.insert(loc=0, column="Session", value=np.ones([bandmat2_df.shape[0], 1]) * sn)
        stageholder2.append(bandmat2_df)
    
    stageholder1_df = pd.concat(stageholder1, ignore_index=True)
    metricdata["p1"].update({stage:stageholder1_df})

    stageholder2_df = pd.concat(stageholder2, ignore_index=True)
    metricdata["p2"].update({stage:stageholder2_df})



In [7]:
key = 'baseline'
metricdata['p2'][key]

,Session,TP9_Theta,TP9_Alpha,TP9_Beta,TP9_Gamma,TP9_TEI,TP10_Theta,TP10_Alpha,TP10_Beta,TP10_Gamma,...,AF7_Alpha,AF7_Beta,AF7_Gamma,AF7_TEI,AF8_Theta,AF8_Alpha,AF8_Beta,AF8_Gamma,AF8_TEI,AlphaAsymmetry
0,2.0,107.857989,39.670831,48.369653,3.916555,0.327866,112.207803,52.927122,42.411146,5.093728,...,119.836344,9823.099762,1596.480988,39.813561,163.120128,46.108627,224.367446,88.718330,1.072355,-0.955127
1,2.0,12.932609,7.394526,11.116274,2.144393,0.546869,7.667636,12.533940,7.996881,3.456010,...,268.694175,10284.751066,2196.194624,23.167672,9.840266,19.204566,156.477437,76.711412,5.387445,-2.638426
2,2.0,101.829254,46.882069,22.769765,3.077339,0.153114,68.867036,35.365307,31.534303,3.951768,...,508.706086,12029.292592,1937.321527,18.104105,84.887083,85.670443,196.555124,84.694455,1.152427,-1.781363
3,2.0,21.676255,6.358914,15.262258,2.266166,0.544397,19.208582,11.669237,5.873886,1.896186,...,267.595485,10184.287619,1628.860423,28.276717,32.796859,53.197427,919.751349,306.800369,10.695494,-1.615466
4,2.0,34.448914,12.586980,10.396740,1.456984,0.221038,9.665723,6.395702,8.319943,3.464219,...,368.395920,9529.150780,2030.760385,16.206539,133.025968,157.398253,1425.970672,491.987917,4.909958,-0.850379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,42.0,29.846933,8.230000,14.883995,1.656715,0.390893,31.684722,9.609119,41.827011,1.381989,...,151.553687,10049.486386,2803.422736,36.329791,5.652882,3.826108,24.382702,0.989610,2.572289,-3.679092
896,42.0,8.917953,6.844392,12.320126,2.236902,0.781618,7.224199,2.174620,15.048656,1.589100,...,1235.648562,6742.826688,4061.063919,4.481650,6.279737,3.299585,7.216372,1.433002,0.753328,-5.925555
897,42.0,161.153907,17.680744,11.035715,1.496859,0.061709,169.653349,15.802781,8.704256,2.329936,...,495.211550,11389.819383,1922.171932,12.529796,11.444211,2.486816,4.333551,1.747400,0.311072,-5.293982
898,42.0,71.555595,13.334139,27.793068,2.444677,0.327402,71.256441,16.709629,23.462884,3.339082,...,741.614698,8295.299410,2811.316583,9.269470,1.864158,4.375826,4.352336,3.247374,0.697492,-5.132735


In [8]:

def bootstrap_ci_median(x, n_boot=10000, ci=95, random_state=1):
    rng = np.random.default_rng(random_state)
    x = np.asarray(x)
    x = x[~np.isnan(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    med = np.median(x)

    boot_meds = np.empty(n_boot)
    n = len(x)

    for i in range(n_boot):
        sample = rng.choice(x, size=n, replace=True)
        boot_meds[i] = np.median(sample)

    alpha = (100 - ci) / 2
    ci_low, ci_high = np.percentile(
        boot_meds, [alpha, 100 - alpha]
    )

    return med, ci_low, ci_high


In [9]:
groups = {
    "Frontal_Theta": ["AF7_Theta", "AF8_Theta"],
    "Frontal_Alpha": ["AF7_Alpha", "AF8_Alpha"],
    "Frontal_Beta": ["AF7_Beta", "AF8_Beta"],
    "Frontal_Gamma": ["AF7_Gamma", "AF8_Gamma"],
    "Frontal_TEI": ["AF7_TEI", "AF8_TEI"]
}

p1_bysession_dict = {}

for stage in stages:
    
    probe = metricdata['p1'][stage].copy()

    for new_col, cols in groups.items():
        probe[new_col] = probe[cols].mean(axis=1)

    probe = probe.drop(columns=sum(groups.values(), []))

    probe_median = probe.groupby("Session").median()

    p1_bysession_dict.update({stage:probe_median})
    


p1_bysession_df = pd.concat(p1_bysession_dict, axis=1)

In [10]:
## P1
p1_summ = []

for col in p1_bysession_df.columns:
    med, lo, hi = bootstrap_ci_median(p1_bysession_df[col])
    p1_summ.append({
        "feature": col,
        "median": med,
        "ci_low": lo,
        "ci_high": hi
    })

p1_summ_df = pd.DataFrame(p1_summ)
p1_summ_df


,feature,median,ci_low,ci_high
0,"(baseline, TP9_Theta)",13.327983,6.583978,27.354638
1,"(baseline, TP9_Alpha)",9.976170,6.870031,16.792257
2,"(baseline, TP9_Beta)",22.567680,13.168219,39.357446
3,"(baseline, TP9_Gamma)",9.722143,4.237636,23.561428
4,"(baseline, TP9_TEI)",0.860195,0.626764,1.620767
5,"(baseline, TP10_Theta)",9.544361,5.916656,14.125484
6,"(baseline, TP10_Alpha)",9.262748,5.547378,13.795690
7,"(baseline, TP10_Beta)",17.566908,11.347661,29.338843
8,"(baseline, TP10_Gamma)",7.360734,4.264981,16.362152
9,"(baseline, TP10_TEI)",0.879301,0.567059,1.552945


In [11]:
p2_bysession_dict = {}
p2_metrics_dict = {}

for stage in stages:
    
    probe = metricdata['p2'][stage].copy()

    for new_col, cols in groups.items():
        probe[new_col] = probe[cols].mean(axis=1)

    probe = probe.drop(columns=sum(groups.values(), []))

    probe_median = probe.groupby("Session").median()

    p2_bysession_dict.update({stage:probe_median})
    
    p2_metrics_dict.update({stage:probe_median.median()})

p2_bysession_df = pd.concat(p2_bysession_dict, axis=1)
p2_metrics_df = pd.concat(p2_metrics_dict, axis=1)

In [12]:
## P2
p2_summ = []

for col in p2_bysession_df.columns:
    med, lo, hi = bootstrap_ci_median(p2_bysession_df[col])
    p2_summ.append({
        "feature": col,
        "median": med,
        "ci_low": lo,
        "ci_high": hi
    })

p2_summ_df = pd.DataFrame(p2_summ)
p2_summ_df

,feature,median,ci_low,ci_high
0,"(baseline, TP9_Theta)",16.728363,10.745334,30.285871
1,"(baseline, TP9_Alpha)",11.121838,8.197302,19.440904
2,"(baseline, TP9_Beta)",17.827491,13.692148,48.963322
3,"(baseline, TP9_Gamma)",5.442080,3.854887,29.331011
4,"(baseline, TP9_TEI)",0.524138,0.507836,1.368688
5,"(baseline, TP10_Theta)",14.937471,11.829211,23.869465
6,"(baseline, TP10_Alpha)",10.968758,9.266271,15.795194
7,"(baseline, TP10_Beta)",18.097110,13.033048,46.133582
8,"(baseline, TP10_Gamma)",6.991612,3.496280,25.237395
9,"(baseline, TP10_TEI)",0.709601,0.504723,1.338738


In [13]:
p2_summ_df['median'][5]

np.float64(14.937470690786213)

In [14]:
## P1

stages = ["baseline", "shared", "individual"]

combs = list(combinations(stages, 2))

feats = p2_metrics_df.index.to_list()


rows = []
pvals = []

for st1, st2 in combs:
    for f in feats:
        x = p1_bysession_df[(st1, f)]
        y = p1_bysession_df[(st2, f)]
        # Wilcoxon
        stat, p = wilcoxon(y, x, zero_method='wilcox', alternative='two-sided')

        pvals.append(p)
        
        # Simple effect size
        med_diff = np.median(y - x)

        # med_diff = np.median()
        rows.append({
        "comp": st2 + ' - ' + st1,
        "feature": f,
        "median_diff_state": med_diff,
        "wilcoxon_stat": stat,
        "p": p
    })
res = pd.DataFrame(rows)

corrected_list = []
for st1, st2 in combs:
    paired = res[res["comp"] == st2 + " - " + st1].copy()
    reject, qvals, _, _ = multipletests(paired["p"].values, alpha=0.05, method="fdr_bh")
    paired["q_fdr_bh"] = qvals
    paired["signif_q<0.05"] = reject
    corrected_list.append(paired)
corrected_df = pd.concat(corrected_list, ignore_index=True)

In [15]:
corrected_df

,comp,feature,median_diff_state,wilcoxon_stat,p,q_fdr_bh,signif_q<0.05
0,shared - baseline,TP9_Theta,5.512329,12.0,0.130859,0.256250,False
1,shared - baseline,TP9_Alpha,6.608076,5.0,0.019531,0.156250,False
2,shared - baseline,TP9_Beta,12.517811,12.0,0.130859,0.256250,False
3,shared - baseline,TP9_Gamma,7.886173,11.0,0.105469,0.256250,False
4,shared - baseline,TP9_TEI,-0.179194,20.0,0.492188,0.562500,False
5,shared - baseline,TP10_Theta,2.858131,17.0,0.322266,0.396635,False
6,shared - baseline,TP10_Alpha,2.070980,13.0,0.160156,0.256250,False
7,shared - baseline,TP10_Beta,10.236324,8.0,0.048828,0.195312,False
8,shared - baseline,TP10_Gamma,6.900416,8.0,0.048828,0.195312,False
9,shared - baseline,TP10_TEI,-0.090727,22.0,0.625000,0.625000,False


In [16]:
## P2

stages = ["baseline", "shared", "individual"]

combs = list(combinations(stages, 2))

feats = p2_metrics_df.index.to_list()


rows = []
pvals = []

for st1, st2 in combs:
    for f in feats:
        x = p2_bysession_df[(st1, f)]
        y = p2_bysession_df[(st2, f)]
        # Wilcoxon
        stat, p = wilcoxon(y, x, zero_method='wilcox', alternative='two-sided')

        pvals.append(p)
        
        # Simple effect size
        med_diff = np.median(y - x)

        # med_diff = np.median()
        rows.append({
        "comp": st2 + ' - ' + st1,
        "feature": f,
        "median_diff_state": med_diff,
        "wilcoxon_stat": stat,
        "p": p
    })
res = pd.DataFrame(rows)

corrected_list = []
for st1, st2 in combs:
    paired = res[res["comp"] == st2 + " - " + st1].copy()
    reject, qvals, _, _ = multipletests(paired["p"].values, alpha=0.05, method="fdr_bh")
    paired["q_fdr_bh"] = qvals
    paired["signif_q<0.05"] = reject
    corrected_list.append(paired)
corrected_df = pd.concat(corrected_list, ignore_index=True)
corrected_df

,comp,feature,median_diff_state,wilcoxon_stat,p,q_fdr_bh,signif_q<0.05
0,shared - baseline,TP9_Theta,-4.257365,14.0,0.193359,0.890625,False
1,shared - baseline,TP9_Alpha,-0.946346,20.0,0.492188,0.890625,False
2,shared - baseline,TP9_Beta,-0.614242,21.0,0.556641,0.890625,False
3,shared - baseline,TP9_Gamma,0.853219,27.0,1.000000,1.000000,False
4,shared - baseline,TP9_TEI,0.110132,18.0,0.375000,0.890625,False
5,shared - baseline,TP10_Theta,-1.115519,17.0,0.322266,0.890625,False
6,shared - baseline,TP10_Alpha,-0.651455,21.0,0.556641,0.890625,False
7,shared - baseline,TP10_Beta,-0.499735,22.0,0.625000,0.909091,False
8,shared - baseline,TP10_Gamma,0.017500,25.0,0.845703,0.966518,False
9,shared - baseline,TP10_TEI,0.000387,26.0,0.921875,0.983333,False


# Connectivity

In [17]:
fs = 256
winsz = 4
ovlap = 0.5

sessions = [2, 12, 14, 21, 27, 28, 31, 39, 41, 42]
# sessions = [2]

stages = ["baseline", "shared", "individual"]
# stages = ["baseline"]

# Freq bands of interest
min_freq = 4
max_freq = 30

# Provide the freq points
freqs = np.linspace(min_freq, max_freq, int((max_freq - min_freq) * 4 + 1))

# The dictionary with frequencies are converted to tuples for the function
fmin = tuple(np.arange(min_freq, max_freq, 1))
fmax = tuple(np.arange(min_freq, max_freq, 1) + 1)

# We will try two different connectivity measurements as an example
connectivity_methods = ["coh", "plv", "pli", "wpli"]
n_con_methods = len(connectivity_methods)

channels = ['TP9', 'TP10', 'AF7', 'AF8']

ch_names = ['TP9_1', 'TP10_1', 'AF7_1', 'AF8_1', 'TP9_2', 'TP10_2', 'AF7_2', 'AF8_2']

indices = ([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3], [4, 5, 6, 7, 4, 5, 6, 7, 4, 5, 6, 7, 4, 5, 6, 7,])

ch_comb = list(product(channels, repeat=2))

conndata = {}

for stage in stages:
    sessdata = {}
    for sn in sessions:
        p1_data = preprodata['p1'][stage][ preprodata['p1'][stage]['Session'] == sn ].drop("Session", axis=1).to_numpy().T
        p2_data = preprodata['p2'][stage][ preprodata['p2'][stage]['Session'] == sn ].drop("Session", axis=1).to_numpy().T
        p1_holder = []
        p2_holder = []

        for i, (sig1, sig2) in enumerate(zip(p1_data, p2_data)):
            p1_holder.append(ctf.timewindow(sig1, fs, winsz, ovlap))
            p2_holder.append(ctf.timewindow(sig2, fs, winsz, ovlap))

        p1_p2_arr = np.vstack((np.array(p1_holder),np.array(p2_holder)))
        p1_p2_formatted = np.transpose(p1_p2_arr, (1, 0, 2))


        info = mne.create_info(ch_names, fs, ch_types='eeg')
        data_epoch = mne.EpochsArray(p1_p2_formatted, info)


        # Pre-allocatate memory for the connectivity matrices
        con_time_array = np.zeros(
            (n_con_methods, p1_p2_formatted.shape[0], len(channels)**2, len(fmin))
        )
        con_time_array[con_time_array == 0] = np.nan  # nan matrix

        # Compute connectivity over time
        con_time = mnec.spectral_connectivity_time(
            data_epoch,
            freqs,
            method=connectivity_methods,
            sfreq=fs,
            mode="cwt_morlet",
            indices=indices,
            fmin=fmin,
            fmax=fmax,
            faverage=True,
            )
            
        for c in range(n_con_methods):
            con_time_array[c] = con_time[c].get_data(output="compact")

        sessdata.update({f'S{sn}':con_time_array})


    conndata[stage] = sessdata


            

            

Not setting metadata
44 matching events found


No baseline correction applied
0 projection items activated
Connectivity computation...
   Processing epoch 1 / 44 ...
   Processing epoch 2 / 44 ...


C:\Users\josee\AppData\Local\Temp\ipykernel_12240\2909440886.py:63: RuntimeWarning: There were no Annotations stored in <EpochsArray | 44 events (all good), 0 – 3.996 s (baseline off), ~2.8 MiB, data loaded,
 '1': 44>, so metadata was not modified.
  con_time = mnec.spectral_connectivity_time(


   Processing epoch 3 / 44 ...
   Processing epoch 4 / 44 ...
   Processing epoch 5 / 44 ...
   Processing epoch 6 / 44 ...
   Processing epoch 7 / 44 ...
   Processing epoch 8 / 44 ...
   Processing epoch 9 / 44 ...
   Processing epoch 10 / 44 ...
   Processing epoch 11 / 44 ...
   Processing epoch 12 / 44 ...
   Processing epoch 13 / 44 ...
   Processing epoch 14 / 44 ...
   Processing epoch 15 / 44 ...
   Processing epoch 16 / 44 ...
   Processing epoch 17 / 44 ...
   Processing epoch 18 / 44 ...
   Processing epoch 19 / 44 ...
   Processing epoch 20 / 44 ...
   Processing epoch 21 / 44 ...
   Processing epoch 22 / 44 ...
   Processing epoch 23 / 44 ...
   Processing epoch 24 / 44 ...
   Processing epoch 25 / 44 ...
   Processing epoch 26 / 44 ...
   Processing epoch 27 / 44 ...
   Processing epoch 28 / 44 ...
   Processing epoch 29 / 44 ...
   Processing epoch 30 / 44 ...
   Processing epoch 31 / 44 ...
   Processing epoch 32 / 44 ...
   Processing epoch 33 / 44 ...
   Processing e

C:\Users\josee\AppData\Local\Temp\ipykernel_12240\2909440886.py:63: RuntimeWarning: There were no Annotations stored in <EpochsArray | 44 events (all good), 0 – 3.996 s (baseline off), ~2.8 MiB, data loaded,
 '1': 44>, so metadata was not modified.
  con_time = mnec.spectral_connectivity_time(


...

   Processing epoch 3 / 119 ...
   Processing epoch 4 / 119 ...
   Processing epoch 5 / 119 ...
   Processing epoch 6 / 119 ...
   Processing epoch 7 / 119 ...
   Processing epoch 8 / 119 ...
   Processing epoch 9 / 119 ...
   Processing epoch 10 / 119 ...
   Processing epoch 11 / 119 ...
   Processing epoch 12 / 119 ...
   Processing epoch 13 / 119 ...
   Processing epoch 14 / 119 ...
   Processing epoch 15 / 119 ...
   Processing epoch 16 / 119 ...
   Processing epoch 17 / 119 ...
   Processing epoch 18 / 119 ...
   Processing epoch 19 / 119 ...
   Processing epoch 20 / 119 ...
   Processing epoch 21 / 119 ...
   Processing epoch 22 / 119 ...
   Processing epoch 23 / 119 ...
   Processing epoch 24 / 119 ...
   Processing epoch 25 / 119 ...
   Processing epoch 26 / 119 ...
   Processing epoch 27 / 119 ...
   Processing epoch 28 / 119 ...
   Processing epoch 29 / 119 ...
   Processing epoch 30 / 119 ...
   Processing epoch 31 / 119 ...
   Processing epoch 32 / 119 ...
   Processing epo

C:\Users\josee\AppData\Local\Temp\ipykernel_12240\2909440886.py:63: RuntimeWarning: There were no Annotations stored in <EpochsArray | 119 events (all good), 0 – 3.996 s (baseline off), ~7.4 MiB, data loaded,
 '1': 119>, so metadata was not modified.
  con_time = mnec.spectral_connectivity_time(


   Processing epoch 3 / 119 ...
   Processing epoch 4 / 119 ...
   Processing epoch 5 / 119 ...
   Processing epoch 6 / 119 ...
   Processing epoch 7 / 119 ...
   Processing epoch 8 / 119 ...
   Processing epoch 9 / 119 ...
   Processing epoch 10 / 119 ...
   Processing epoch 11 / 119 ...
   Processing epoch 12 / 119 ...
   Processing epoch 13 / 119 ...
   Processing epoch 14 / 119 ...
   Processing epoch 15 / 119 ...
   Processing epoch 16 / 119 ...
   Processing epoch 17 / 119 ...
   Processing epoch 18 / 119 ...
   Processing epoch 19 / 119 ...
   Processing epoch 20 / 119 ...
   Processing epoch 21 / 119 ...
   Processing epoch 22 / 119 ...
   Processing epoch 23 / 119 ...
   Processing epoch 24 / 119 ...
   Processing epoch 25 / 119 ...
   Processing epoch 26 / 119 ...
   Processing epoch 27 / 119 ...
   Processing epoch 28 / 119 ...
   Processing epoch 29 / 119 ...
   Processing epoch 30 / 119 ...
   Processing epoch 31 / 119 ...
   Processing epoch 32 / 119 ...
   Processing epo

C:\Users\josee\AppData\Local\Temp\ipykernel_12240\2909440886.py:63: RuntimeWarning: There were no Annotations stored in <EpochsArray | 119 events (all good), 0 – 3.996 s (baseline off), ~7.4 MiB, data loaded,
 '1': 119>, so metadata was not modified.
  con_time = mnec.spectral_connectivity_time(


   Processing epoch 3 / 119 ...
   Processing epoch 4 / 119 ...
   Processing epoch 5 / 119 ...
   Processing epoch 6 / 119 ...
   Processing epoch 7 / 119 ...
   Processing epoch 8 / 119 ...
   Processing epoch 9 / 119 ...
   Processing epoch 10 / 119 ...
   Processing epoch 11 / 119 ...
   Processing epoch 12 / 119 ...
   Processing epoch 13 / 119 ...
   Processing epoch 14 / 119 ...
   Processing epoch 15 / 119 ...
   Processing epoch 16 / 119 ...
   Processing epoch 17 / 119 ...
   Processing epoch 18 / 119 ...
   Processing epoch 19 / 119 ...
   Processing epoch 20 / 119 ...
   Processing epoch 21 / 119 ...
   Processing epoch 22 / 119 ...
   Processing epoch 23 / 119 ...
   Processing epoch 24 / 119 ...
   Processing epoch 25 / 119 ...
   Processing epoch 26 / 119 ...
   Processing epoch 27 / 119 ...
   Processing epoch 28 / 119 ...
   Processing epoch 29 / 119 ...
   Processing epoch 30 / 119 ...
   Processing epoch 31 / 119 ...
   Processing epoch 32 / 119 ...
   Processing epo

In [61]:
# Define estos nombres (si ya los tienes, usa los tuyos)        # len = 4
ch_pairs     = [f"{i}_{j}" for i,j in ch_comb] # len = 16
band_names   = [f"{i}-{j} Hz" for i,j in list(zip(fmin,fmax))]   # len = 26
cols = [f"{i}_{j}" for i,j in list(product(ch_pairs, band_names))]

dfs_by_stage = {}

for stage in stages:
    blocks = []

    for sn in sessions:
        for metric in range(0,4):
            hd = conndata[stage][f'S{sn}'][metric, :, 0, :]
            for ch in range(1,16):
                hd = np.hstack((hd, conndata[stage][f'S{sn}'][metric, :, ch, :]))
            df_sess = pd.DataFrame(hd,columns=cols)
            df_sess.insert(0, "Session", sn)
            df_sess.insert(1, "Metric", connectivity_methods[metric])
            df_sess.insert(2, "Epoch", np.arange(1,hd.shape[0]+1))

            blocks.append(df_sess)

    dfs_by_stage[stage] = pd.concat(blocks, axis=0, ignore_index=True)

In [63]:
dfs_by_stage['shared']

,Session,Metric,Epoch,TP9_TP9_4-5 Hz,TP9_TP9_5-6 Hz,TP9_TP9_6-7 Hz,TP9_TP9_7-8 Hz,TP9_TP9_8-9 Hz,TP9_TP9_9-10 Hz,TP9_TP9_10-11 Hz,...,AF8_AF8_20-21 Hz,AF8_AF8_21-22 Hz,AF8_AF8_22-23 Hz,AF8_AF8_23-24 Hz,AF8_AF8_24-25 Hz,AF8_AF8_25-26 Hz,AF8_AF8_26-27 Hz,AF8_AF8_27-28 Hz,AF8_AF8_28-29 Hz,AF8_AF8_29-30 Hz
0,2,coh,1,0.435331,0.265176,0.126319,0.143081,0.176729,0.204499,0.214510,...,0.230170,0.225131,0.217078,0.208854,0.203443,0.202357,0.204918,0.209225,0.213246,0.215486
1,2,coh,2,0.212922,0.151059,0.141376,0.217118,0.263462,0.236690,0.205429,...,0.264963,0.265633,0.261511,0.255279,0.250419,0.249443,0.252206,0.256462,0.259971,0.261445
2,2,coh,3,0.406830,0.308202,0.211847,0.228272,0.177876,0.138261,0.115878,...,0.107643,0.098460,0.091214,0.085930,0.084277,0.090601,0.108611,0.125108,0.137242,0.145001
3,2,coh,4,0.516067,0.671200,0.549320,0.372926,0.265283,0.194577,0.163077,...,0.121805,0.104854,0.087421,0.070516,0.055088,0.042370,0.034871,0.037032,0.046671,0.051923
4,2,coh,5,0.264742,0.327602,0.302571,0.284069,0.224906,0.199516,0.194595,...,0.237745,0.246595,0.247965,0.240712,0.226640,0.209839,0.194951,0.185344,0.182121,0.184345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4755,42,wpli,115,0.742708,0.663480,0.445052,0.299590,0.233776,0.258054,0.305537,...,0.142260,0.154934,0.170674,0.179790,0.180637,0.174960,0.166472,0.160385,0.159749,0.165584
4756,42,wpli,116,0.305424,0.521597,0.594511,0.579059,0.531222,0.461778,0.363500,...,0.190951,0.180648,0.170307,0.157444,0.141244,0.122751,0.105250,0.097389,0.120481,0.173359
4757,42,wpli,117,0.148723,0.246562,0.423898,0.392979,0.350269,0.254045,0.225348,...,0.598476,0.638944,0.673304,0.696471,0.709300,0.716074,0.716644,0.710855,0.698498,0.673983
4758,42,wpli,118,0.400939,0.350203,0.383555,0.283178,0.210291,0.117187,0.115808,...,0.590592,0.605682,0.621128,0.634186,0.642529,0.646275,0.644928,0.640827,0.635778,0.626924


In [ ]:
# import pickle

# # with open("conndata.pkl", "wb") as f:
# #     pickle.dump(conndata, f)

# with open("conndata.pkl", "rb") as f:
#     conndata = pickle.load(f)